# Assignment 2 - GitHub Issues Collection and Cleaning

## 1. Aim

Collect public GitHub issues and issue comments related to AI coding assistants so we can analyse:

- Text themes: reliability, bugs, productivity, privacy, security, cost, code ownership, trust, and workflow friction.
- Network structure: users connected to issues, users participating in the same issue threads, and issue-label relationships.

In [ ]:
import json
import re
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

def find_project_root():
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "collection_notebooks").exists() and (path / "src").exists():
            return path
    for path in (current, *current.parents):
        nested = path / "social-media-2"
        if (nested / "collection_notebooks").exists() and (nested / "src").exists():
            return nested
    raise FileNotFoundError("Could not find the social-media-2 project root")

PROJECT_ROOT = find_project_root()
project_root_str = str(PROJECT_ROOT.resolve())
if project_root_str not in sys.path:
    sys.path.append(project_root_str)

from src.utils.github_client import githubGet, githubHeaders

In [ ]:
RAW_DIR = PROJECT_ROOT / "data/raw/github"
PROCESSED_DIR = PROJECT_ROOT / "data/processed/github"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

REPOSITORIES = [
    {"owner": "microsoft", "repo": "vscode-copilot-release"},
    {"owner": "openai", "repo": "codex"},
    {"owner": "anthropics", "repo": "claude-code"},
]

ISSUE_STATE = "all"
MAX_ISSUES_PER_REPO = 400
MAX_COMMENTS_PER_ISSUE = 400

# Existing raw files are reused by default. Set to True to pull fresh data from the API.
FORCE_API_PULL = False

expected_raw_files = [
    RAW_DIR / f"github_issues_{repo['owner']}_{repo['repo']}.json"
    for repo in REPOSITORIES
]

print("Notebook working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Raw GitHub directory:", RAW_DIR)
print("Processed GitHub directory:", PROCESSED_DIR)

REQUIRE_GITHUB_TOKEN = True
GITHUB_TOKEN_CONFIGURED = "Authorization" in githubHeaders()
print("GitHub token configured:", GITHUB_TOKEN_CONFIGURED)

raw_files_ready = all(path.exists() for path in expected_raw_files)
print("Existing raw GitHub files ready:", raw_files_ready)

if (FORCE_API_PULL or not raw_files_ready) and REQUIRE_GITHUB_TOKEN and not GITHUB_TOKEN_CONFIGURED:
    raise RuntimeError(
        "GITHUB_TOKEN is not configured. Add it to .env in the project root, "
        "then restart the kernel. Do not put tokens in tracked source files."
    )

## 2. Data Collection Functions

This keeps the GitHub API use deliberately simple:

1. Pull issues from each repository.
2. Ignore pull requests, because GitHub's issues endpoint also returns PRs.
3. Pull comments for each issue.
4. Save one nested raw JSON file per repository.

In [ ]:
def fetch_repo_issues(owner, repo, state="all", max_issues=300):
    url = f"https://api.github.com/repos/{owner}/{repo}/issues"
    all_issues = []
    page = 1

    while len(all_issues) < max_issues:
        params = {
            "state": state,
            "per_page": 100,
            "page": page,
            "sort": "updated",
            "direction": "desc",
        }

        issues = githubGet(url, params=params)
        if not issues:
            break

        for issue in issues:
            if "pull_request" in issue:
                continue

            all_issues.append(issue)
            if len(all_issues) >= max_issues:
                break

        page += 1

    return all_issues


def fetch_issue_comments(owner, repo, issue_number, max_comments=200):
    url = f"https://api.github.com/repos/{owner}/{repo}/issues/{issue_number}/comments"
    all_comments = []
    page = 1

    while len(all_comments) < max_comments:
        params = {
            "per_page": 100,
            "page": page,
        }

        comments = githubGet(url, params=params)
        if not comments:
            break

        for comment in comments:
            all_comments.append(comment)
            if len(all_comments) >= max_comments:
                break

        page += 1

    return all_comments


def fetchGithubIssueData(owner, repo, state="all", maxIssues=300, maxCommentsPerIssue=200, outputFile="githubDataDump.json"):
    issues = fetch_repo_issues(owner, repo, state=state, max_issues=maxIssues)
    issue_threads = []

    for index, issue in enumerate(issues, start=1):
        comments = []

        if issue.get("comments", 0) > 0:
            try:
                comments = fetch_issue_comments(
                    owner,
                    repo,
                    issue["number"],
                    max_comments=maxCommentsPerIssue,
                )
            except Exception as e:
                print(f"Could not fetch comments for {owner}/{repo} issue #{issue['number']}: {e}")

        issue_threads.append({
            "repository": f"{owner}/{repo}",
            "collectionRank": index,
            "issue": issue,
            "comments": comments,
        })

    data = {
        "repository": f"{owner}/{repo}",
        "state": state,
        "issueThreads": issue_threads,
    }

    with open(outputFile, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(issue_threads)} issue threads to {outputFile}")

## 3. Collect Raw GitHub Data

In [ ]:
print("GitHub collection config:")
print("Repositories:", [f"{repo['owner']}/{repo['repo']}" for repo in REPOSITORIES])
print("MAX_ISSUES_PER_REPO:", MAX_ISSUES_PER_REPO)
print("MAX_COMMENTS_PER_ISSUE:", MAX_COMMENTS_PER_ISSUE)

for repo_config in REPOSITORIES:
    owner = repo_config["owner"]
    repo = repo_config["repo"]
    output_file = RAW_DIR / f"github_issues_{owner}_{repo}.json"
    if output_file.exists() and not FORCE_API_PULL:
        print(f"\nUsing existing raw file for {owner}/{repo} -> {output_file}")
        continue

    print(f"\nCollecting {owner}/{repo} -> {output_file}")

    fetchGithubIssueData(
        owner,
        repo,
        state=ISSUE_STATE,
        maxIssues=MAX_ISSUES_PER_REPO,
        maxCommentsPerIssue=MAX_COMMENTS_PER_ISSUE,
        outputFile=str(output_file),
    )

## 4. Flatten Raw JSON into Tables

This creates separate issue and comment tables, plus a combined text table for NLP.

In [ ]:
missing_raw_files = [str(path) for path in expected_raw_files if not path.exists()]
if missing_raw_files:
    raise FileNotFoundError(
        "Missing expected raw GitHub files:\n"
        + "\n".join(missing_raw_files)
        + "\nRun the collection cell before flattening."
    )

issue_rows = []
comment_rows = []
text_rows = []

for json_file in expected_raw_files:
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    for thread in data.get("issueThreads", []):
        issue = thread["issue"]
        repository = thread["repository"]
        issue_key = f"{repository}#{issue['number']}"
        labels = [label.get("name", "") for label in issue.get("labels", [])]

        issue_rows.append({
            "repository": repository,
            "issueKey": issue_key,
            "issueNumber": issue["number"],
            "issueId": issue["id"],
            "title": issue.get("title", ""),
            "body": issue.get("body") or "",
            "state": issue.get("state"),
            "author": issue.get("user", {}).get("login"),
            "createdAt": issue.get("created_at"),
            "updatedAt": issue.get("updated_at"),
            "closedAt": issue.get("closed_at"),
            "commentCount": issue.get("comments", 0),
            "labels": ";".join(labels),
            "htmlUrl": issue.get("html_url"),
        })

        text_rows.append({
            "repository": repository,
            "issueKey": issue_key,
            "recordType": "issue",
            "recordId": issue["id"],
            "author": issue.get("user", {}).get("login"),
            "createdAt": issue.get("created_at"),
            "text": f"{issue.get('title', '')}\n\n{issue.get('body') or ''}",
            "url": issue.get("html_url"),
        })

        for comment in thread.get("comments", []):
            comment_rows.append({
                "repository": repository,
                "issueKey": issue_key,
                "issueNumber": issue["number"],
                "issueTitle": issue.get("title", ""),
                "issueAuthor": issue.get("user", {}).get("login"),
                "commentId": comment.get("id"),
                "commentAuthor": comment.get("user", {}).get("login"),
                "commentBody": comment.get("body") or "",
                "commentCreatedAt": comment.get("created_at"),
                "commentUpdatedAt": comment.get("updated_at"),
                "commentUrl": comment.get("html_url"),
            })

            text_rows.append({
                "repository": repository,
                "issueKey": issue_key,
                "recordType": "comment",
                "recordId": comment.get("id"),
                "author": comment.get("user", {}).get("login"),
                "createdAt": comment.get("created_at"),
                "text": comment.get("body") or "",
                "url": comment.get("html_url"),
            })

issues_df = pd.DataFrame(issue_rows)
comments_df = pd.DataFrame(comment_rows)
text_df = pd.DataFrame(text_rows)

issues_df.to_csv(PROCESSED_DIR / "GitHubIssuesRawFlattened.csv", index=False)
comments_df.to_csv(PROCESSED_DIR / "GitHubCommentsRawFlattened.csv", index=False)
text_df.to_csv(PROCESSED_DIR / "GitHubDiscussionTextRaw.csv", index=False)

print("Issues:", issues_df.shape)
print("Comments:", comments_df.shape)
print("Text records:", text_df.shape)

issues_df.head()

## 5. Basic Data Checks

In [ ]:
print("Repositories:")
print(issues_df["repository"].value_counts().to_string())

print("\nIssue states:")
print(issues_df["state"].value_counts(dropna=False).to_string())

print("\nTop authors by issue count:")
print(issues_df["author"].value_counts().head(20).to_string())

print("\nTop comment authors:")
if len(comments_df):
    print(comments_df["commentAuthor"].value_counts().head(20).to_string())
else:
    print("No comments collected")

In [ ]:
if len(issues_df):
    issues_df["createdAt"] = pd.to_datetime(issues_df["createdAt"], errors="coerce")
    issues_df["updatedAt"] = pd.to_datetime(issues_df["updatedAt"], errors="coerce")
    print("Issue date range:", issues_df["createdAt"].min(), "to", issues_df["createdAt"].max())

if len(comments_df):
    comments_df["commentCreatedAt"] = pd.to_datetime(comments_df["commentCreatedAt"], errors="coerce")
    print("Comment date range:", comments_df["commentCreatedAt"].min(), "to", comments_df["commentCreatedAt"].max())

    try:
        import matplotlib.pyplot as plt
        comments_df["commentCreatedAt"].dt.year.value_counts().sort_index().plot(kind="bar", figsize=(8, 5))
        plt.title("GitHub Comments by Year")
        plt.xlabel("Year")
        plt.ylabel("Number of Comments")
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("matplotlib is not installed, skipping the year plot.")

## 6. Text Cleaning

Keep the raw Markdown text, but add a cleaned plain-text-ish field for first-pass NLP. This does not need to be perfect yet; it just removes the worst formatting noise.

In [ ]:
def clean_github_text(text):
    text = str(text)
    text = re.sub(r"```.*?```", " ", text, flags=re.DOTALL)
    text = re.sub(r"`([^`]*)`", r"\1", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[#>*_\-\[\]()]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


text_df = text_df[text_df["text"].notna()].copy()
text_df["textClean"] = text_df["text"].apply(clean_github_text)
text_df["textLower"] = text_df["textClean"].str.lower()
text_df["textLength"] = text_df["textClean"].str.len()
text_df["createdAt"] = pd.to_datetime(text_df["createdAt"], errors="coerce")

text_df = text_df[text_df["textLength"] > 0].copy()
text_df = text_df.drop_duplicates(subset=["repository", "recordType", "recordId", "textClean"]).copy()

text_df.to_csv(PROCESSED_DIR / "GitHubDiscussionTextCleaned.csv", index=False)

print("Clean text records:", text_df.shape)
text_df.head()

## 7. NLP Filtering

Automated GitHub messages and very low-information comments are filtered out before NLP. The full cleaned text table is still kept, while the filtered table is used for sentiment and topic modelling.


In [ ]:
original_text_rows = len(text_df)

author = text_df["author"].fillna("")
text = text_df["text"].fillna("")
clean = text_df["textClean"].fillna("")

bot_author = author.str.contains(r"\[bot\]|bot$", case=False, regex=True)
auto_terms = [
    "possible duplicate",
    "duplicate issues",
    "automatically closed as a duplicate",
    "marked as stale",
    "closed this issue because",
    "this issue has been automatically",
    "generated " + "with",
    "co-" + "authored-by",
]
auto_message = text.str.contains("|".join(auto_terms), case=False, regex=True)
short_text = clean.str.len() < 25
pile_on = clean.str.match(
    r"^\s*(\+1|same|same here|me too|bump|following|any update|yes|still broken)[\s!?.+-]*$",
    case=False,
    na=False,
)
very_long_comment = (text_df["recordType"] == "comment") & (text_df["textLength"] > 3000)

exclude_for_nlp = bot_author | auto_message | short_text | pile_on | very_long_comment

text_df["include_for_nlp"] = ~exclude_for_nlp
text_df["exclude_reason"] = ""
text_df.loc[bot_author, "exclude_reason"] = "bot author"
text_df.loc[auto_message, "exclude_reason"] = "automated message"
text_df.loc[short_text, "exclude_reason"] = "short text"
text_df.loc[pile_on, "exclude_reason"] = "low information response"
text_df.loc[very_long_comment, "exclude_reason"] = "very long comment"

filtered_text_df = text_df[text_df["include_for_nlp"]].copy()
filtered_text_df.to_csv(PROCESSED_DIR / "GitHubDiscussionTextFiltered.csv", index=False)

print(f"Kept {len(filtered_text_df):,} of {original_text_rows:,} text records for NLP")
print("Excluded records by reason:")
print(text_df.loc[~text_df["include_for_nlp"], "exclude_reason"].value_counts().to_string())

text_df = filtered_text_df


## 8. Relevance Checks

These keywords are a quick sanity check for whether the issue discussion is useful for the research question. We can adjust this list later.

In [ ]:
relevance_keywords = [
    "copilot",
    "chat",
    "agent",
    "review",
    "bug",
    "error",
    "crash",
    "slow",
    "hallucinat",
    "privacy",
    "security",
    "trust",
    "productivity",
    "cost",
    "subscription",
    "model",
    "context",
]

for keyword in relevance_keywords:
    count = text_df["textLower"].str.contains(re.escape(keyword), regex=True).sum()
    print(f"{keyword}: {count}")

In [ ]:
word_counter = Counter()

for text in text_df["textLower"]:
    words = re.findall(r"[a-z][a-z0-9_]{2,}", str(text))
    word_counter.update(words)

common_words = word_counter.most_common(30)
common_words

## 9. Network Edge Tables


In [ ]:
user_issue_rows = []

for _, row in issues_df.iterrows():
    if pd.notna(row.get("author")):
        user_issue_rows.append({
            "source": row["author"],
            "target": row["issueKey"],
            "relationship": "opened_issue",
            "weight": 1,
        })

for _, row in comments_df.iterrows():
    if pd.notna(row.get("commentAuthor")):
        user_issue_rows.append({
            "source": row["commentAuthor"],
            "target": row["issueKey"],
            "relationship": "commented_on_issue",
            "weight": 1,
        })

user_issue_edges = pd.DataFrame(user_issue_rows)

if len(user_issue_edges):
    user_issue_edges = (
        user_issue_edges
        .groupby(["source", "target", "relationship"], as_index=False)
        .agg(weight=("weight", "sum"))
    )

user_issue_edges.to_csv(PROCESSED_DIR / "GitHubUserIssueEdges.csv", index=False)

print("User-issue edges:", user_issue_edges.shape)
user_issue_edges.head()

In [ ]:
issue_label_rows = []

for _, row in issues_df.iterrows():
    labels = str(row.get("labels", "")).split(";") if pd.notna(row.get("labels")) else []
    for label in labels:
        label = label.strip()
        if label:
            issue_label_rows.append({
                "source": row["issueKey"],
                "target": label,
                "relationship": "has_label",
                "weight": 1,
            })

issue_label_edges = pd.DataFrame(issue_label_rows)
issue_label_edges.to_csv(PROCESSED_DIR / "GitHubIssueLabelEdges.csv", index=False)

print("Issue-label edges:", issue_label_edges.shape)
issue_label_edges.head()